[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/moving_mirror/moving_mirror.ipynb)

# Waves off a Moving Mirror

This example solves the scalar wave equation across an entire triangulated spacetime strip at once rather than stepping a spatial grid forward in time. A localized source starting from quiet initial rows radiates strictly inside its future light cone, and a mirror advancing into a cavity reflects an incident pulse into a narrower, higher-amplitude wave packet whose extra energy comes from the work done by the moving boundary.

Spacetime events are modelled as intrinsic antivectors, and joining two boundary displacements of a spacetime triangle forms its signed patch `(sides[0] & sides[1]) / 2`. Dividing each opposite side displacement by twice that patch gives the ***simplex gradient*** `sides / (2 * patch)` without invoking a metric, while the Lorentzian signature enters once through `Plane.dual()` pulled back as `gradient.adjugate()(medium(gradient))`; reflecting that dual across an observer's simultaneity plane `-(field & (t >> Plane.dual()(field))) / 2` converts the indefinite action density into positive wave energy density.

In coordinate finite-element exterior calculus on a deforming $(x, t)$ domain, assembling the wave stiffness matrix $K_{pq} = \sum_T |T|\,(\nabla\lambda_p)^{\mathsf T} J_T^{-1}\,\operatorname{diag}(1, -1)\,J_T^{-\mathsf T}\nabla\lambda_q$ from barycentric hat functions $\lambda_p(x, t)$ requires per-triangle $2\times 2$ Jacobian inversions $J_T^{-1}$ and determinant signs, while the observer energy integrates $\frac{1}{2}\int ((\partial_t\phi)^2 + (\partial_x\phi)^2)\,dx$ separately from the indefinite Lagrangian $\frac{1}{2}((\partial_x\phi)^2 - (\partial_t\phi)^2)$.

In [ ]:
# In Colab, install numga and its example dependencies from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from collections.abc import Iterator

%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext
from numga.sparse import Sparse, spdiag
from numga.topology import Topology
from examples.relativity.moving_mirror import render, scenarios
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("x+t-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Plane = ga.gatype.vector()                                                     # Vector+
Point = ga.gatype.antivector()                                                 # Antivector-
Patch = ga.gatype.antibivector()                                               # Antibivector-
Amount = ga.gatype.antiscalar()                                                # Antiscalar-
WaveMap = ga.gatype((Amount, Scalar))                                          # Amount- <- Scalar+
GradientMap = ga.gatype((Plane, Scalar))                                       # Plane+ <- Scalar+
triangle = Topology(3, [[0, 1, 2]])

# Unit spatial and future-timelike displacements pair to one with `mv.x` and `mv.t`.
space, time = mv.x.dual(), -mv.t.dual()                                        # [] Point-, Point-

## Simplex gradients and causal light-cone wakes

On any spacetime triangle, dividing each opposite side displacement by twice the signed patch gives a metric-free simplex gradient plane and circulating edge 1-forms. Pulling `Plane.dual()` back through the assembled simplex gradient builds a Lorentzian wave operator whose quiet initial rows send a localized pulse strictly along its forward null rays.

In [ ]:
n_triangle_subdivisions = scenarios.n_triangle_subdivisions
n_wake_space = scenarios.n_wake_space
n_wake_time = scenarios.n_wake_time
wake_duration = scenarios.wake_duration
source_event = scenarios.source_x * space + scenarios.source_t * time          # [] Point-
source_width = scenarios.source_width

In [ ]:
def corner_interpolation(corners: Point, points: Point) -> tuple[Scalar, Plane, Patch]:
    """Dividing opposite side displacements of `corners` (`Point-[3]`) by twice the signed patch gives the simplex gradients (`[3] Plane+`), corner weights (`[3, ...] Scalar+`), and patch (`[] Patch-`)."""
    sides = (triangle.boundary[2, 1] * (triangle.d[0] * corners)).batch()[:, 0]  # [3] Point-
    patch = (sides[0] & sides[1]) / 2                                          # [] Patch-
    # Dividing each opposite worldline displacement by twice the signed patch gives a simplex gradient plane
    # that drops by one from its corner event to the opposite edge without using a metric.
    gradients = sides / (2 * patch)                                            # [3] Plane+
    corner_events = corners.batch()                                            # [3] Point-
    weights = 1 + (gradients[:, None] & (points[None, :] - corner_events[:, None]))  # [3, ...] Scalar+
    return weights, gradients, patch


def edge_whitney_forms(corners: Point, points: Point) -> Plane:
    """Dividing displacements from `corners` (`Point-[3]`) to `points` (`[...] Point-`) by twice the signed patch gives the three edge 1-forms (`[3, ...] Plane+`)."""
    sides = (triangle.boundary[2, 1] * (triangle.d[0] * corners)).batch()[:, 0]  # [3] Point-
    patch = (sides[0] & sides[1]) / 2                                          # [] Patch-
    corner_events = corners.batch()                                            # [3] Point-
    return (points[None, :] - corner_events[:, None]) / (2 * patch)            # [3, ...] Plane+


corners, points = scenarios.single_triangle(space, time, n_triangle_subdivisions)  # Point-[3], [...] Point-
weights, gradients, patch = corner_interpolation(corners, points)
edge_forms = edge_whitney_forms(corners, points)                               # [3, ...] Plane+

In [ ]:
def spacetime_operators(topology: Topology, vertices: Point) -> tuple[GradientMap, WaveMap, Scalar]:
    """Assembling the simplex gradient across `topology` and pulling back the Lorentzian dual gives the gradient (`(Plane+ <- Scalar+)[F <- V]`), wave operator (`(Amount- <- Scalar+)[V <- V]`), and areas (`Scalar+[F]`)."""
    edges = topology.d[0] * vertices                                           # Point-[E]
    sides = topology.boundary[2, 1] * edges                                    # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                        # Patch-[F]
    areas = patches.norm()                                                     # Scalar+[F]
    corner_gradients = sides / (2 * patches)                                   # [3] Plane+[F]
    gradient = (spdiag(corner_gradients) * topology.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[F <- V]
    # Pulling the Lorentzian plane dual back through the simplex gradient builds a wave operator with null characteristics.
    medium = areas * Plane.dual()                                              # (Point- <- Plane+)[F]
    wave_operator = gradient.adjugate()(medium(gradient))                      # (Amount- <- Scalar+)[V <- V]
    return gradient, wave_operator, areas


def causal_wake(
    topology: Topology,
    vertices: Point,
    wave_operator: WaveMap,
    areas: Scalar,
    n_space: int,
    n_time: int,
    source_event: Point,
    source_width: float,
) -> Scalar:
    """Solving the spacetime wave equation with two quiet initial rows radiates the causal light-cone wake (`Scalar+[V]`) from `source_event`."""
    displacements = vertices - source_event                                    # Point-[V]
    delay = mv.t & displacements                                               # Scalar+[V]
    distance_squared = (mv.x & displacements).squared() + delay.squared()      # Scalar+[V]
    # A time-derivative Gaussian source radiates two sharp pulses along the forward 45-degree null rays.
    profile = (delay / source_width) * (-(distance_squared / (2 * source_width**2))).exp()  # Scalar+[V]
    source = (profile * areas.sites.mean()).dual()                             # Amount-[V]

    grid = np.arange(n_time * n_space).reshape(n_time, n_space)
    interior = topology.reading(grid[1:-1, 1:-1].ravel(), 0).adjugate() * Scalar  # (Scalar+ <- Scalar+)[V <- V_interior]
    future = topology.reading(grid[2:, 1:-1].ravel(), 0).adjugate() * Scalar   # (Scalar+ <- Scalar+)[V <- V_future]
    return future(interior.adjugate()(wave_operator(future)).solve(interior.adjugate()(source)))  # Scalar+[V]


wake_topology, wake_vertices, _ = scenarios.spacetime_strip(
    space, time, n_wake_space, n_wake_time, np.ones(n_wake_time), wake_duration,
)
_, wake_wave_operator, wake_areas = spacetime_operators(wake_topology, wake_vertices)
wake = causal_wake(
    wake_topology,
    wake_vertices,
    wake_wave_operator,
    wake_areas,
    n_wake_space,
    n_wake_time,
    source_event,
    source_width,
)
save_figure(
    render.light_cone_figure(
        corners, points, gradients, edge_forms, wake_topology, wake_vertices, wake, source_event,
    ),
    "moving_mirror_light_cone",
);

The left panel shows level lines of one corner's affine interpolation together with its Euclidean normal in blue and its Lorentzian dual reflected across the dashed light cone in orange, the middle panel traces the hyperbolic dual streamlines of an edge 1-form around its opposite corner, and the right panel shows the forward wake spreading along the dashed 45-degree null rays before reflecting off the stationary walls.

In [ ]:
print("Spacetime triangle area:", patch.norm().to_array())
print("Simplex gradient Lorentzian squares:", (gradients & Plane.dual()(gradients)).to_array())

## Relativistic Doppler compression on a moving-mirror strip

Deforming the right boundary worldline inward along a smooth timelike curve models an advancing mirror while keeping flat Minkowski spacetime on every triangle. Enforcing zero potential along both walls reflects a right-moving pulse into a compressed packet with shortened wavelength, amplified time derivative, and higher total wave energy.

In [ ]:
n_mirror_space = scenarios.n_mirror_space
n_mirror_time = scenarios.n_mirror_time
mirror_duration = scenarios.mirror_duration
initial_length = scenarios.initial_length
stroke = scenarios.stroke
inflection_time = scenarios.inflection_time
transition_time = scenarios.transition_time
pulse_centre = scenarios.pulse_centre
pulse_width = scenarios.pulse_width
wavelength = scenarios.wavelength
before_row = scenarios.before_row
after_row = scenarios.after_row

In [ ]:
def moving_mirror_wave(
    topology: Topology,
    vertices: Point,
    gradient: GradientMap,
    wave_operator: WaveMap,
    areas: Scalar,
    n_space: int,
    n_time: int,
    dt: float,
    pulse_centre: float,
    pulse_width: float,
    wavelength: float,
) -> tuple[Scalar, Plane, Scalar, Scalar]:
    """Solving for a right-moving wave packet reflecting off the moving boundary gives the potential (`Scalar+[V]`), gradient (`Plane+[F]`), nodal wave rate (`Scalar+[V]`), and slab energies (`[n_time - 1] Scalar+`)."""
    # Pairing with the null plane `mv.x - mv.t` reads the phase of the right-moving incident wave at every event.
    phase = ((mv.x - mv.t) & vertices) - pulse_centre                          # Scalar+[V]
    envelope = (-(phase / pulse_width).squared()).exp()                        # Scalar+[V]
    carrier = (2 * np.pi * phase / wavelength).cos()                           # Scalar+[V]

    grid = np.arange(n_time * n_space).reshape(n_time, n_space)
    seed = topology.reading(grid[:2, 1:-1].ravel(), 0)                         # Scalar+[V_seed <- V]
    initial = seed.adjugate() * (seed * (envelope * carrier))                  # Scalar+[V]

    interior = topology.reading(grid[1:-1, 1:-1].ravel(), 0).adjugate() * Scalar  # (Scalar+ <- Scalar+)[V <- V_interior]
    future = topology.reading(grid[2:, 1:-1].ravel(), 0).adjugate() * Scalar   # (Scalar+ <- Scalar+)[V <- V_future]
    rhs = -interior.adjugate()(wave_operator(initial))                         # Amount-[V_interior]
    potential = initial + future(interior.adjugate()(wave_operator(future)).solve(rhs))  # Scalar+[V]

    field = gradient(potential)                                                # Plane+[F]
    face_rate = field & time                                                   # Scalar+[F]
    to_vertices = topology.incidence[2, 0].sum(axis=0).adjugate()              # Scalar+[V <- F]
    wave_rate = (to_vertices * (areas * face_rate)) / (to_vertices * areas)    # Scalar+[V]

    # Reflecting the Lorentzian dual across the observer's slice flips the temporal sign into positive Hamiltonian energy density.
    energy_density = -(field & (mv.t >> Plane.dual()(field))) / 2              # Scalar+[F]
    slab_indices = topology.elements[2].min(axis=-1) // n_space                # [F]
    to_slabs = Sparse.selection(slab_indices, n_time - 1).adjugate()           # Scalar+[n_time - 1 <- F]
    slice_energies = (to_slabs * (areas * energy_density)).batch() / dt        # [n_time - 1] Scalar+
    return potential, field, wave_rate, slice_energies


mirror_times = np.linspace(0, mirror_duration, n_mirror_time)
wall_positions = scenarios.mirror_trajectory(
    mirror_times, initial_length, stroke, inflection_time, transition_time,
)
mirror_topology, mirror_vertices, dt = scenarios.spacetime_strip(
    space, time, n_mirror_space, n_mirror_time, wall_positions, mirror_duration,
)
mirror_gradient, mirror_wave_operator, mirror_areas = spacetime_operators(
    mirror_topology, mirror_vertices,
)
_, _, wave_rate, slice_energies = moving_mirror_wave(
    mirror_topology,
    mirror_vertices,
    mirror_gradient,
    mirror_wave_operator,
    mirror_areas,
    n_mirror_space,
    n_mirror_time,
    dt,
    pulse_centre,
    pulse_width,
    wavelength,
)
save_figure(
    render.doppler_figure(
        mirror_topology,
        mirror_vertices,
        n_mirror_space,
        n_mirror_time,
        wave_rate,
        slice_energies,
        before_row,
        after_row,
    ),
    "moving_mirror_doppler",
);

The left panel shows the worldsheet wave rate and deforming strip lines with dashed simultaneity slices before reflection in blue and after reflection in orange, the middle panel compares the broad incident wave-rate profile against the taller, narrower reflected packet, and the right panel tracks the step in total slab wave energy as the pulse bounces off the advancing mirror.

In special-relativistic wave notation, reflection off a mirror advancing at speed $v$ compresses wavelength and scales both $\partial_t\phi$ and integrated pulse energy by $(1+v)/(1-v)$, which here varies smoothly across the packet as the mirror decelerates.

In [ ]:
wave_rate_grid = wave_rate.batch().reshape(n_mirror_time, n_mirror_space)      # [n_mirror_time, n_mirror_space] Scalar+
before_peak = wave_rate_grid[before_row].abs().max()                           # [] Scalar+
after_peak = wave_rate_grid[after_row].abs().max()                             # [] Scalar+
print("Peak wave-rate ratio (after / before):", (after_peak / before_peak).to_array())
print("Wave energy ratio    (after / before):", (slice_energies[after_row] / slice_energies[before_row]).to_array())

## Sweeping simultaneity slices across the cavity

Streaming simultaneity row indices upward across the solved worldsheet links the global spacetime history on the left to the instantaneous spatial cavity and advancing mirror on the right.

In [ ]:
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def cavity_slices(n_time: int, n_frames: int) -> Iterator[int]:
    """Yields `n_frames` evenly spaced simultaneity row indices sweeping across the `n_time` worldsheet rows."""
    for row in np.linspace(4, n_time - 4, n_frames, dtype=int):
        yield int(row)


recorded_rows = show_states(
    cavity_slices(n_mirror_time, n_frames),
    render.cavity_animation,
    mirror_topology,
    mirror_vertices,
    n_mirror_space,
    n_mirror_time,
    wave_rate,
    name="moving_mirror_cavity",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

As the orange simultaneity bar rises up the worldsheet, the right-moving pulse meets the inward-moving dark wall and rebounds leftward with visibly tighter oscillations and higher amplitude.

In [ ]:
# checks
assert len(recorded_rows) == n_frames
corner_events = corners.batch()                                                # [3] Point-
corner_weights, _, _ = corner_interpolation(corners, corner_events)
assert ((corner_weights - np.eye(3)).abs() < 1e-10).all()
assert ((weights.sum(axis=0) - 1).abs() < 1e-10).all()
gradient_sum = gradients.sum(axis=0)                                           # [] Plane+
assert ((gradient_sum & space).abs() < 1e-10) and ((gradient_sum & time).abs() < 1e-10)

tails, heads = triangle.corners[2, 1]
classic_forms = (
    weights[tails] * gradients[heads, None] - weights[heads] * gradients[tails, None]
) * triangle.orientation[1].T
form_residual = edge_forms - classic_forms                                     # [3, ...] Plane+
assert ((form_residual & space).abs() < 1e-10).all()
assert ((form_residual & time).abs() < 1e-10).all()
sides = (triangle.boundary[2, 1] * (triangle.d[0] * corners)).batch()[:, 0]    # [3] Point-
midpoints = (triangle.incidence[2, 1] * (triangle.incidence[1, 0].mean(axis=0) * corners)).batch()[:, 0]  # [3] Point-
midpoint_readings = edge_whitney_forms(corners, midpoints) & sides[None, :]    # [3, 3] Scalar+
assert ((midpoint_readings - np.eye(3)).abs() < 1e-10).all()

right_null, left_null = mv.x - mv.t, mv.x + mv.t                               # [] Plane+, Plane+
assert (right_null & Plane.dual()(right_null)).abs() < 1e-10
assert (left_null & Plane.dual()(left_null)).abs() < 1e-10

wake_displacements = wake_vertices.batch() - source_event
wake_delays = mv.t & wake_displacements
outside_cone = ((mv.x & wake_displacements).abs() > wake_delays + 0.14) | (wake_delays < -0.10)
wake_batch = wake.batch()
assert wake_batch[outside_cone].abs().max() < 0.02 * wake_batch.abs().max()
assert after_peak > 1.85 * before_peak
assert slice_energies[after_row] > 2.15 * slice_energies[before_row]